# YOLOv11 Chess Pieces Detection - Google Colab Training

This notebook is designed to train the YOLOv11m Chess Piece Detection model on Google Colab.
It mounts Google Drive to retrieve/backup the dataset, converts the ChessReD COCO JSON format to standard YOLO format, and runs the training loop using the direct Ultralytics API.

## Step 1: Environment Setup
We will check if the runtime is Google Colab, mount Google Drive, adjust the working directory to the project root, and install the required dependencies.

In [1]:
# Check if running in Google Colab
import os
import sys
import google.colab

print("Running in Google Colab. Mounting Google Drive...")
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Install dependencies if running in Colab
print("Installing project dependencies...")
# Install extra required libraries for Ultralytics & training
!pip install ultralytics draccus gdown tqdm pillow albumentations pyyaml

Installing project dependencies...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 12.6 MB/s eta 0:00:00


## Step 2: Dataset Preparation
To avoid downloading the heavy dataset from the internet every time the Colab runtime restarts, this step will:
1. Check if the dataset already exists on your Google Drive.
2. If it exists on Drive, it copies it directly to the Colab local SSD and extracts it (takes only a few seconds).
3. If it does not exist on Drive, it automatically downloads it from the internet (annotations and images zip) and saves a backup copy to your Google Drive (`chess_pieces_detection/datasets/`) so that subsequent runs can load it instantly without internet downloading.

In [3]:
import shutil
import zipfile
import urllib.request
from pathlib import Path
import gdown

# Define paths (Adjust DRIVE_DIR to your Drive project datasets folder)
DRIVE_DIR = Path("/content/drive/MyDrive/chess_pieces_detection/datasets")
LOCAL_DIR = Path("datasets")
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

# URL resources
ANN_URL = "https://data.4tu.nl/file/99b5c721-280b-450b-b058-b2900b69a90f/3cae6364-daca-4967-b426-1e4b68cdb64c"
ZIP_GD_ID = "1jxmFxjOy0qefdCZ_x3DMNtsvAK4LojEw"

# 1. Prepare annotations.json
local_ann = LOCAL_DIR / "annotations.json"
drive_ann = DRIVE_DIR / "annotations.json"

if not local_ann.exists():
    if drive_ann.exists():
        print("Copying annotations.json from Google Drive...")
        shutil.copy(drive_ann, local_ann)
    else:
        print("Downloading annotations.json from internet...")
        urllib.request.urlretrieve(ANN_URL, local_ann)

# 2. Prepare and extract images
local_images = LOCAL_DIR / "images"
local_zip = LOCAL_DIR / "preprocessed_images.zip"
drive_zip = DRIVE_DIR / "preprocessed_images.zip"

if not (local_images.exists() and any(local_images.iterdir())):
    if drive_zip.exists():
        print("Copying preprocessed_images.zip from Google Drive...")
        shutil.copy(drive_zip, local_zip)
    else:
        print("Downloading preprocessed_images.zip from internet...")
        gdown.download(id=ZIP_GD_ID, output=str(local_zip), quiet=False)
        
    print("Extracting images dataset...")
    with zipfile.ZipFile(local_zip, 'r') as zip_ref:
        zip_ref.extractall(LOCAL_DIR)
    local_zip.unlink()  # Remove zip file to save local VM space
    print("Extraction completed.")
else:
    print("Dataset images are already present and extracted.")


Copying preprocessed_images.zip from Google Drive...
Extracting images dataset...
Extraction completed.


## Step 3: Convert ChessReD COCO dataset format to standard YOLO format
Ultralytics YOLO standard training API works out of the box with the standard YOLODataset format (directory structure of `images/split/` and `labels/split/` along with separate `.txt` labels for each image).
We will run a conversion script to transform ChessReD COCO annotations into standard YOLO text labels, link images into splits without taking extra disk space, and generate a new `dataset.yaml` with correct absolute paths.

In [4]:
import json
import yaml
from collections import defaultdict
import os
import shutil
from pathlib import Path

# Source and destination paths
COCO_JSON = Path("datasets/annotations.json")
IMAGES_DIR = Path("datasets/images")
YOLO_DIR = Path("datasets/yolo_dataset")

def convert_chessred_coco_to_yolo(coco_json_path, images_dir_path, output_yolo_dir):
    coco_json_path = Path(coco_json_path)
    images_dir_path = Path(images_dir_path)
    output_yolo_dir = Path(output_yolo_dir)
    
    print(f"Starting conversion from COCO JSON {coco_json_path} to YOLO format...")
    
    # Read annotations
    with open(coco_json_path, 'r') as f:
        coco = json.load(f)
        
    # Build category map (excluding 'empty')
    valid_cats = [c for c in coco.get("categories", []) if c.get("name", "") != "empty"]
    valid_cats = sorted(valid_cats, key=lambda c: c["id"])
    cat_id_to_cls = {cat["id"]: i for i, cat in enumerate(valid_cats)}
    
    # Build class name map for dataset.yaml
    class_names = {i: cat["name"] for i, cat in enumerate(valid_cats)}
    
    # Create YOLO directory structure
    splits = ['train', 'val', 'test']
    for split in splits:
        (output_yolo_dir / "images" / split).mkdir(parents=True, exist_ok=True)
        (output_yolo_dir / "labels" / split).mkdir(parents=True, exist_ok=True)
        
    # Get image splits map
    split_img_ids = {}
    splits_data = coco.get("splits", {})
    for split in splits:
        if split in splits_data:
            split_img_ids[split] = set(splits_data[split]["image_ids"])
        else:
            split_img_ids[split] = set()
            
    # Group annotations by image_id
    img_to_anns = defaultdict(list)
    anns_data = coco.get("annotations", {})
    pieces_list = anns_data.get("pieces", []) if isinstance(anns_data, dict) else (anns_data if isinstance(anns_data, list) else [])
    
    for ann in pieces_list:
        img_to_anns[ann["image_id"]].append(ann)
        
    # Process images
    converted_count = 0
    skipped_count = 0
    
    for img_info in coco.get("images", []):
        img_id = img_info["id"]
        # height & width in ChessReD coco json
        h, w = img_info["height"], img_info["width"]
        
        # Determine split
        img_split = 'train'  # Default fallback
        for split in splits:
            if img_id in split_img_ids[split]:
                img_split = split
                break
                
        # Source image file path
        rel_path = img_info.get("path", img_info.get("file_name", ""))
        src_img = images_dir_path / rel_path
        if not src_img.exists() and rel_path.startswith("images/"):
            src_img = images_dir_path / rel_path[len("images/"):]
        if not src_img.exists():
            src_img = images_dir_path.parent / rel_path
            
        if not src_img.exists():
            skipped_count += 1
            continue
            
        # Destination path
        img_filename = src_img.name
        dst_img = output_yolo_dir / "images" / img_split / img_filename
        dst_lbl = output_yolo_dir / "labels" / img_split / f"{src_img.stem}.txt"
        
        # Create hardlink/copy for images (fast and space efficient)
        if not dst_img.exists():
            try:
                os.link(src_img, dst_img)
            except Exception:
                shutil.copy(src_img, dst_img)
                
        # Convert bounding boxes and write label file
        bboxes = []
        for ann in img_to_anns.get(img_id, []):
            cat_id = ann.get("category_id")
            if cat_id not in cat_id_to_cls:
                continue
            cls_idx = cat_id_to_cls[cat_id]
            
            bbox_raw = ann.get("bbox")
            if bbox_raw is None or len(bbox_raw) != 4:
                continue
                
            box = [float(val) for val in bbox_raw]
            
            # Convert ChessReD [x_min, y_min, w, h] to YOLO center normalized [x_c, y_c, w_n, h_n]
            x_center = box[0] + box[2] / 2.0
            y_center = box[1] + box[3] / 2.0
            
            x_center_norm = x_center / w
            y_center_norm = y_center / h
            w_norm = box[2] / w
            h_norm = box[3] / h
            
            # Clip bounds to be safe
            x_center_norm = max(0.0, min(1.0, x_center_norm))
            y_center_norm = max(0.0, min(1.0, y_center_norm))
            w_norm = max(0.0, min(1.0, w_norm))
            h_norm = max(0.0, min(1.0, h_norm))
            
            bboxes.append(f"{cls_idx} {x_center_norm:.6f} {y_center_norm:.6f} {w_norm:.6f} {h_norm:.6f}")
            
        with open(dst_lbl, 'w') as lf:
            lf.write('\n'.join(bboxes))
            
        converted_count += 1
        
    print(f"Conversion complete! Converted: {converted_count}, Skipped: {skipped_count}")
    return class_names

if COCO_JSON.exists() and IMAGES_DIR.exists():
    # Convert and retrieve class name map
    classes = convert_chessred_coco_to_yolo(COCO_JSON, IMAGES_DIR, YOLO_DIR)
    
    # Save the dataset.yaml file inside the converted directory
    yolo_yaml_path = YOLO_DIR / "dataset.yaml"
    yolo_yaml_data = {
        "path": str(YOLO_DIR.resolve()),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "nc": len(classes),
        "names": classes
    }
    with open(yolo_yaml_path, 'w') as f:
        yaml.safe_dump(yolo_yaml_data, f, default_flow_style=False)
        
    print(f"Successfully generated YOLO dataset config at {yolo_yaml_path}")
else:
    print("Error: Source dataset files not found!")

Starting conversion from COCO JSON datasets/annotations.json to YOLO format...
Conversion complete! Converted: 10800, Skipped: 0
Successfully generated YOLO dataset config at datasets/yolo_dataset/dataset.yaml


## Step 4: Run YOLOv11m Fine-Tuning using Ultralytics API
We load the pretrained YOLOv11m model checkpoint and train it directly using the standard Ultralytics API `YOLO(model).train(...)` pointing to our converted `dataset.yaml` config.

In [ ]:
import torch
from ultralytics import YOLO
from pathlib import Path

DRIVE_DIR = Path("/content/drive/MyDrive/chess_pieces_detection/datasets")

# Check GPU availability
device_name = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device_name}")
if device_name == "cuda":
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
# Save results to 'runs/' folder inside your Drive project directory
drive_runs = str(DRIVE_DIR.parent / "runs")
# Initialize YOLO model
model = YOLO("drive/MyDrive/chess_pieces_detection/runs/chess_detection_yolo11m-2/weights/last.pt")

# Start training using direct Ultralytics API
results = model.train(
    data="datasets/yolo_dataset/dataset.yaml",
    project=drive_runs,
    epochs=15,
    batch=16,            # Adjust based on Colab GPU VRAM (e.g. 16/32, or -1 for AutoBatch)
    imgsz=1024,
    device=device_name,
    optimizer="auto",
    name="chess_detection_yolo11m",
    resume=False,       # Change to True if you need to resume interrupted training
    exist_ok=False,
)

Using device: cuda
GPU Model: Tesla T4


Ultralytics 8.4.105 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=datasets/yolo_dataset/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=drive/MyDrive/chess_pieces_detection/runs/chess_detection_yolo11m-2/weights/last.pt, mo